# Read source-level and external-participant evidence

This notebook reproduces the primary paired ranking calculation and presents forecasting, masking, nuisance controls, and external transfer at their correct evaluation units.

In [ ]:
from pathlib import Path
import sys, json, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT = next((p for p in candidates if (p/'worldmodels/v2_data.py').is_file()), None)
if PROJECT is None:
    PROJECT = next(p/'experiments/aaai-worldmodels' for p in candidates if (p/'experiments/aaai-worldmodels/worldmodels/v2_data.py').is_file())
sys.path.insert(0, str(PROJECT))
REPOSITORY = PROJECT.parents[1]
from worldmodels.v2_data import load_cohort
from worldmodels import v2_evaluation as ev
from worldmodels import v2_models as models
from worldmodels.v2_runner import window_errors, verify_freeze
COHORT = PROJECT/'artifacts/runs/reference-v2-cohort-v1'
RUN = PROJECT/'artifacts/runs/reference-v2-core-v1'
TRANSFER = PROJECT/'artifacts/runs/reference-v2-transfer-v1'
FIGURES = PROJECT/'artifacts/executed-reference-v2/figures'
FIGURES.mkdir(parents=True, exist_ok=True)
USED_INPUTS={}
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def record_input(path):
    path=Path(path).resolve()
    key=str(path.relative_to(REPOSITORY)) if path.is_relative_to(REPOSITORY) else str(path)
    USED_INPUTS[key]=sha(path)
def read_json(path):
    path = Path(path)
    assert path.is_file(), f'Required completed artifact is absent: {path}. Read docs/reference_v2/README.md for prerequisites.'
    record_input(path)
    return json.loads(path.read_text())
def read_csv(path, **kwargs):
    record_input(path)
    return pd.read_csv(path, **kwargs)
def artifact(name, directory=RUN):
    path = directory/name
    manifest = read_json(directory/'artifact_hashes.json')
    assert name in manifest and path.is_file(), f'Unrecorded or missing artifact: {path}'
    assert sha(path) == manifest[name], f'Artifact hash differs from the completed run: {path}'
    record_input(path)
    return path
def completed_run():
    result = read_json(artifact('results.json'))
    assert result['status']=='complete' and not result.get('test_only', False)
    return result
def save_figure(figure, name):
    figure.savefig(FIGURES/(name+'.svg'), bbox_inches='tight')
    figure.savefig(FIGURES/(name+'.png'), dpi=150, bbox_inches='tight')
freeze = verify_freeze()
record_input(PROJECT/'docs/reference_v2/protocol_freeze.json')
x, valid, metadata, actual_times = load_cohort(COHORT, verify=True)
cohort_identity=read_json(COHORT/'identity.json');record_input(COHORT/'cohort.csv')
for name in cohort_identity['prepared_inputs']:
    record_input(REPOSITORY/cohort_identity['prepared_directory']/name)
assert x.shape == (len(metadata),24,16,2) and len(metadata)==2815
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':11,'svg.fonttype':'none','axes.spines.top':False,'axes.spines.right':False})
print('Verified real observations:',len(metadata),'windows;',metadata.source_id.nunique(),'source videos.')
print('Observation units: pelvis-relative 2D coordinates divided by a prefix torso scale. Times are decoder seconds.')


## Compare normal-source forecasts on common targets

We first ask whether the models predict normal movement more accurately. The reported MSE averages observed coordinates within each leg, then legs, windows, sequences, and sources. It covers all eight future samples. A positive paired difference means the listed method reduces error relative to its comparator.

In [ ]:
result=completed_run()
means=pd.DataFrame(result['normal_forecast'])
display(means.query('horizon==8')[['arm','mse','ci95','normal_sources']])
contrasts=pd.DataFrame(result['forecast_contrasts'])
display(contrasts[contrasts.method.isin(['ridge_posture_velocity','mlp_anatomical_tubes','mlp_motion_balanced'])][['method','reference','estimate','ci95','relative_reduction']])
mean_priors=read_json(artifact('results.json',PROJECT/'artifacts/runs/reference-v2-mean-prior-posthoc-v1'))
display(pd.DataFrame(mean_priors['comparisons'])[['method','reference','method_mse','reference_mse','estimate','ci95']])
print('The two untuned normal-mean forecast controls were declared after the core outcomes were known. Their separate post hoc identity preserves that history.')


## Recompute the primary anomaly endpoint

An anomaly score is evaluated by whether it ranks annotated abnormal sources above normal sources. The primary comparison uses only pairs scored within the same outer fold. Its average weights each fold by the number of abnormal-normal source pairs. We compare the locked calibrated detector with low motion and separately with an identically calibrated posture-only model.

In [ ]:
source=read_csv(artifact('source_anomaly_scores.csv'))
primary=ev.paired_within_fold_bootstrap(source,'calibrated_two_sided','low_motion',reps=5000,seed=5701)
np.testing.assert_allclose(primary['estimate'],result['primary_anomaly']['estimate'],rtol=0,atol=1e-12)
np.testing.assert_allclose(primary['ci95'],result['primary_anomaly']['ci95'],rtol=0,atol=1e-12)
display(pd.DataFrame(primary['folds']))
display({k:primary[k] for k in ['candidate_auroc','reference_auroc','estimate','ci95','uncertainty']})
print(f"The calibrated model has within-fold AUROC {primary['candidate_auroc']:.3f}, compared with {primary['reference_auroc']:.3f} for low motion.")
print(f"The paired difference is {primary['estimate']:.3f}, with conditional interval [{primary['ci95'][0]:.3f}, {primary['ci95'][1]:.3f}].")
print('This is a result about incremental ranking in the declared source population. It does not measure clinical diagnostic accuracy.')
display(result['posture_calibration_contrast'])


## Retain simple alternative explanations

The current-posture, movement, and observation-quality controls are part of the result. Calibrating an error does not prove that its signal comes from coordination. Pooled rankings also compare scores across separately fitted folds and therefore need a comparability assumption that within-fold pairs avoid.

In [ ]:
ranking=[]
for row in result['rankings']:
    if row['score'] not in ['calibrated_two_sided','posture_two_sided','low_motion','posture_distance','motion_quality_distance','negative_prefix_motion','high_prefix_missingness']: continue
    ranking.append({'score':row['score'],'within_fold_AUROC':row['within_fold']['pair_weighted_auroc'],
                    'pooled_AUROC':row['pooled']['auroc'],'average_precision':row['pooled']['average_precision'],
                    'prevalence':row['pooled']['prevalence']})
display(pd.DataFrame(ranking))
display(read_csv(artifact('condition_scores.csv')).query("score in ['calibrated_two_sided','posture_two_sided','low_motion']"))


## Read the external population separately

The frozen source-trained model is applied to a different dataset without target-domain fitting or calibration. Errors and prefix covariates are aggregated within recordings and then within participant before each normal-reference transform. The fifteen frozen component scores are then averaged. This differs from averaging scores from individual short windows.

In [ ]:
transfer=read_json(artifact('results.json',TRANSFER))
assert transfer['status']=='complete'
participants=read_csv(artifact('participant_scores.csv',TRANSFER))
display(transfer['participants_by_label'])
print('External windows, recordings, participants:',transfer['windows'],transfer['recordings'],transfer['participants'])
display(transfer['primary_anomaly'])
display(transfer['posture_calibration_contrast'])
external_contrast=transfer['primary_anomaly']
print(f"On the external participant-coded population, the calibrated and low-motion AUROCs are {external_contrast['candidate_auroc']:.3f} and {external_contrast['reference_auroc']:.3f}.")
print('Movement amount may itself contain clinically relevant information; conditioning on it does not automatically create a better detector.')
print('The external outer_fold=0 marker denotes one external population, not a new cross-validation fit.')
assert participants.source_id.nunique()==len(participants)


## Separate uncertainty from new independent evidence

The source and participant bootstrap intervals condition on saved fitted models. Two additional normal-source partitions measure sensitivity of compact forecast fits to training membership. Those correlated partitions and the three optimization seeds do not increase the number of participants.

In [ ]:
retraining=read_json(artifact('retraining_summary.json'))
display(retraining)
print('Intervals describe saved-fit source variation; repeated-partition changes describe an additional, separately reported fitting sensitivity.')
print('Neither internal source grouping nor a different dataset name proves that every evaluated person is independent.')

# Retain the exact inputs used by this independent-kernel execution.
input_path=PROJECT/'artifacts/executed-reference-v2/inputs/24_source_and_external_evaluation.json'
input_path.parent.mkdir(parents=True,exist_ok=True)
input_path.write_text(json.dumps({'path_base':'repository root','inputs':USED_INPUTS},indent=2)+'\n')
